# Week 7 · 04 Anomaly detection

A factory makes thousands of good parts and a handful of faulty ones, and nobody has labelled the faulty ones in advance: they're rare, and every fault is different. So instead of learning "good vs faulty", **anomaly detection** learns what *normal* looks like and flags whatever doesn't fit. This notebook has three ways to measure "doesn't fit" (a Gaussian's Mahalanobis distance, an isolation forest, an autoencoder's reconstruction error), how to set the threshold, and how to evaluate a detector honestly when anomalies are rare. Fault detection in machines and buildings is one of Prof. Yoon's main application areas, and precision–recall (your Week 2 gap) comes back here properly.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **anomaly (outlier)**: a data point that doesn't look like the rest: a faulty part, a failing machine's reading, a fraudulent payment.
- **anomaly score**: a number per data point, larger = more unusual. Every method in this notebook is a different way to compute it.
- **threshold**: the score above which a point is flagged as an anomaly.
- **true / false positive**: a flagged point that really is an anomaly / a flagged point that is actually normal (a false alarm).
- **precision**: of the flagged points, the fraction that are real anomalies: TP / (TP + FP).
- **recall (true positive rate)**: of the real anomalies, the fraction flagged: TP / (TP + FN).
- **false positive rate**: of the normal points, the fraction flagged: FP / (FP + TN).
- **unsupervised / one-class**: trained on unlabelled data, or on normal data only.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from figures import *
import checks; checks.load(globals(), "04")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** The notebook in one picture.

In [ ]:
fig_anomaly_map()

| Section | What | Why it's here |
|---|---|---|
| 0 | recap, and booklet 8.5 (DBSCAN's core points) from memory | DBSCAN's noise points were your first anomaly detector |
| E | an autoencoder finds a scratch on a part | start from the end |
| 1 | the setting | why we learn "normal" instead of "good vs faulty" |
| 2 | a Gaussian score: the Mahalanobis distance | a proposition, a hand calculation, and why per-sensor checks miss faults |
| 3 | isolation forest | an algorithm card: few random cuts = unusual |
| 4 | choosing the threshold | from the normal data's scores |
| 5 | evaluating a detector | confusion matrix, ROC, precision–recall, both built by hand |
| 6 | the autoencoder's score, evaluated | `02`'s reconstruction error on the part factory |
| 7 | which method when, and the deep ones | Deep SVDD and PatchCore in one paragraph each |
| 8 | exercises | the block's hand-written parts |

## 0. Recap first · 10 min, closed book

**Quick check:** In DBSCAN, which points get the label $-1$, and why are they candidates for anomalies?

<details><summary>Answer</summary>

Points that are neither core points nor within $\varepsilon$ of one: noise, points in no dense region. [Week 4 · 04 §5 ↗](../week4/04_hierarchical_dbscan_gmm.ipynb)

</details>

**Quick check:** Precision and recall: the two formulas, and which one goes down when you flag more points?

<details><summary>Answer</summary>

Precision $= TP/(TP + FP)$, recall $= TP/(TP + FN)$. Flagging more raises recall (more of the real ones caught) and usually lowers precision (more false alarms). [Week 3 · 01 §9 ↗](../week3/01_logistic_regression_in_pictures.ipynb)

</details>

**Quick check:** What does PR-AUC (average precision) summarize?

<details><summary>Answer</summary>

The precision–recall curve over all thresholds, in one number; random guessing gives the fraction of positives, not 0.5. [Week 3 · 01 §10 ↗](../week3/01_logistic_regression_in_pictures.ipynb)

</details>

**Quick check:** The covariance matrix $S$ of two features with standard deviations 4 and 1 and correlation 0.8?

<details><summary>Answer</summary>

$\begin{pmatrix} 16 & 3.2 \\ 3.2 & 1 \end{pmatrix}$: variances on the diagonal, $0.8 \cdot 4 \cdot 1 = 3.2$ off it. [Week 4 · 02 §4 ↗](../week4/02_pca.ipynb)

</details>

**Quick check:** `02`: what does an autoencoder do with an input unlike anything in its training data?

<details><summary>Answer</summary>

It rebuilds it badly: it can only produce what it learned to produce, so the reconstruction error is large. §6 uses that error as the score. [02 §3 ↗](02_autoencoders.ipynb)

</details>

**Exercise 1 · Booklet 8.5: DBSCAN's core points.** Return a boolean array `(n,)`: point $i$ is a core point when at least `min_samples` points (counting itself) lie within distance `eps`. Use `NearestNeighbors(radius=eps).fit(X).radius_neighbors(X, return_distance=False)`, which returns, for each point, the array of its neighbours' indices.

**Example**: what goes in, and what should come out:
```python
X = np.array([[0.0], [0.1], [0.2], [5.0]])
core_points(X, eps=0.15, min_samples=3)    # → array([False,  True, False, False])   only 0.1 has 3 within 0.15
```

In [ ]:
def core_points(X, eps, min_samples):
    pass  # your code

In [ ]:
check(1)

## E. Start from the end: an autoencoder finds a scratch

The setting for this notebook's main example: a production line makes rings, and a good ring looks like a handwritten **0**. Every MNIST 0 is a good part. `ae_zeros.pt` is `02`'s autoencoder ($k = 8$) trained only on the 5,923 training zeros. The test parts: 880 good zeros, 100 zeros with a **scratch** (a bright bar painted on), and 100 wrong parts (other digits). `zeros_data()` and `recon_scores()` come from `figures.py`:

In [ ]:
model = load_ae_zeros()
Z_tr, good, scratched, others = zeros_data()          # all (n, 784), pixels in [0, 1]
print(Z_tr.shape, good.shape, scratched.shape, others.shape)
# → torch.Size([5923, 784]) torch.Size([880, 784]) torch.Size([100, 784]) torch.Size([100, 784])

**The score** of a part is its reconstruction error (`02`'s exercise 3). **The threshold** is set from the training zeros alone: the value that only 1% of good training parts exceed. Anything above it is flagged:

In [ ]:
s_train = recon_scores(model, Z_tr)                     # (5923,): one score per training part
thr = np.quantile(s_train, 0.99)                        # 99% of good parts score below this
print(round(thr, 4))
for name, X in [("good", good), ("scratched", scratched), ("other digits", others)]:
    s = recon_scores(model, X)
    print(name, round((s > thr).mean(), 3))             # the fraction flagged
# → 0.0425
#   good 0.016
#   scratched 0.42
#   other digits 0.89

🖼 **Picture.** Five test parts. Row 2: the autoencoder rebuilds every input as a clean 0, since that's all it knows. Row 3: the squared error per pixel lights up exactly where the input differs from a clean 0: the scratch is **located**, not just detected.

In [ ]:
fig_ae_defect_maps()

So: 1.6% false alarms on good parts, 42% of scratches caught, 89% of wrong parts caught. **The rest of this notebook answers:** why learn "normal" at all (§1), what other scores exist (§2–3), how to set the threshold (§4), and whether "42% caught, 1.6% false alarms" is good, and how to say it in one number (§5–6).

## 1. The setting: learn what normal looks like

**The problem with classification.** A classifier needs many labelled examples of each class. Faults are **rare** (a few in thousands), **varied** (every fault is different: a scratch, a dent, a wrong part), and often **unknown in advance** (tomorrow's fault may be new). There's nothing to learn "faulty" from.

**The idea.** Normal data is plentiful and fairly consistent. Learn what it looks like, then score every new point by how badly it fits: a large score means unusual. A threshold turns scores into flags. Every method in this notebook is this recipe; they differ only in how they measure "fits".

| Step | What | In §E |
|---|---|---|
| 1. fit | learn normal from (mostly) normal data, no labels | train an AE on good zeros |
| 2. score | a number per new point, larger = more unusual | reconstruction error |
| 3. threshold | flag points whose score is above it | 99th percentile of the training scores |
| 4. evaluate | if a few labelled anomalies exist, measure precision and recall | §5–6 |

**A second example, small enough to compute by hand.** A machine has two sensors, temperature and vibration. When it runs hotter it also shakes more (correlation 0.8). We have 500 normal readings and five readings from a faulty machine, P to T:

🖼 **Picture.** Left: the readings. Right: each sensor alone. R (hot but calm) and S (cool but shaking) are within the normal range of each sensor separately.

In [ ]:
fig_sensor_data()

**The catch.** A check per sensor ("is the temperature unusual? is the vibration unusual?") misses R and S: neither value is unusual on its own; only the **combination** is. A good score has to look at the features together. That's what the Mahalanobis distance does.

## 2. A Gaussian score: the Mahalanobis distance

**The idea.** Fit a Gaussian to the normal data (its mean $\mu$ and covariance $S$), and score a point by how many "standard deviations" it is from the mean, measured **along the shape of the cloud**: a step along the correlation is cheap, a step across it is expensive.

$$
d_M(x) = \sqrt{(x - \mu)^\top S^{-1} (x - \mu)}
$$

| Symbol | Shape | In words |
|---|---|---|
| $x$ | (d,) | the point to score |
| $\mu$ | (d,) | the mean of the normal data |
| $S$ | (d, d) | the covariance matrix of the normal data |
| $S^{-1}$ | (d, d) | its inverse: it undoes the stretching and the tilt of the cloud |
| $d_M$ | one number | the distance in units of the cloud's own spread |

**A worked example by hand**, with the machine's true parameters $\mu = (60, 4)$, $S = \begin{pmatrix} 16 & 3.2 \\ 3.2 & 1 \end{pmatrix}$, and reading R $= (70, 3)$. **Step 1:** $x - \mu = (10, -1)$. **Step 2:** the inverse of a 2×2 matrix: $\det S = 16 \cdot 1 - 3.2^2 = 5.76$, so $S^{-1} = \frac{1}{5.76}\begin{pmatrix} 1 & -3.2 \\ -3.2 & 16 \end{pmatrix}$. **Step 3:**

$$
d_M^2 = \frac{1}{5.76}\Big(1 \cdot 10^2 \;+\; 2 \cdot (-3.2) \cdot 10 \cdot (-1) \;+\; 16 \cdot (-1)^2\Big) = \frac{100 + 64 + 16}{5.76} = 31.25, \qquad d_M = 5.59
$$

**Compare** with standardizing each sensor separately: $z = (10/4,\; -1/1) = (2.5, -1)$, length $\sqrt{6.25 + 1} = 2.69$. Per sensor, R is unremarkable (under 3); together, it's 5.6 standard deviations out. The middle term (64 of the 180) is what the correlation adds: R is hot, so it **should** be shaking more, and it isn't.

In [ ]:
X, A = sensor_data()                              # X: (500, 2) normal readings; A: (5, 2) faulty P..T
mu, S = X.mean(axis=0), np.cov(X.T)               # estimated from the normal data (np.cov wants features as rows)
D = A - mu
d = np.sqrt(np.sum(D @ np.linalg.inv(S) * D, axis=1))    # row i: sqrt(D_i^T S^-1 D_i)
print(d.round(2))                                 # → [4.54 3.96 5.7  5.74 6.22]   P, Q, R, S, T
from scipy.spatial.distance import mahalanobis
print(round(mahalanobis(A[2], mu, np.linalg.inv(S)), 2))  # → 5.7   R again, from SciPy

R comes out at 5.70 rather than the 5.59 computed by hand, because here $\mu$ and $S$ are estimated from the 500 readings instead of being the machine's true values.

🖼 **Picture.** Contours of equal distance 1, 2 and 3. Left: standardized per sensor, the contours are axis-aligned and R and S fall inside the 3-contour. Right: Mahalanobis, the contours follow the cloud, and every faulty reading is outside.

In [ ]:
fig_mahalanobis()

**Proposition.** The Mahalanobis distance is the ordinary Euclidean distance after **whitening**: if $S = L L^\top$ (for example $L = S^{1/2}$), then $d_M(x) = \lVert L^{-1}(x - \mu) \rVert$, and for Gaussian data the whitened point $L^{-1}(x - \mu)$ is $\mathcal{N}(0, I)$.

**What it says, in words.** Undo the cloud's stretch and tilt, and it becomes a round ball of spread 1; then measure plain distance. **The proof.** $S^{-1} = (LL^\top)^{-1} = L^{-\top} L^{-1}$, so $(x - \mu)^\top S^{-1}(x - \mu) = (L^{-1}(x - \mu))^\top (L^{-1}(x - \mu)) = \lVert L^{-1}(x - \mu)\rVert^2$. And if $x \sim \mathcal{N}(\mu, S)$, then $L^{-1}(x - \mu)$ has mean 0 and covariance $L^{-1} S L^{-\top} = L^{-1} L L^\top L^{-\top} = I$. $\blacksquare$

**Remarks.** (1) **A threshold from theory:** for Gaussian data, $d_M^2$ is a sum of $d$ squared standard normals, a $\chi^2_d$ variable; for $d = 2$, 99% of normal points have $d_M^2 < 9.21$, i.e. $d_M < 3.03$. (2) It needs $S^{-1}$: with many features and few samples $S$ is singular; use shrinkage (`sklearn.covariance.LedoitWolf`) or PCA first. (3) It assumes one roughly elliptical cloud; for several clusters, use a Gaussian mixture's density (Week 4) or §3. (4) Outliers in the training data inflate $S$; `sklearn.covariance.MinCovDet` estimates it robustly.

**In one sentence:** the Mahalanobis distance measures how far a point is from the mean in units of the data's own spread and correlation, so it catches combinations that are unusual even when each value is not.

## 3. Isolation forest

**The idea** (Liu et al., 2008). Don't model normal data at all. Unusual points are **few and different**, so random cuts separate them from everything else quickly; a normal point, deep inside the crowd, needs many cuts before it's alone.

🖼 **Picture.** Random cuts (a random sensor, then a random value between the min and max of the points still in the box) until the chosen point is alone. The faulty reading P is isolated after 2 cuts; a typical reading needed 11.

In [ ]:
fig_isolation()

**Algorithm card.**

- **Goal:** a score per point that's large when the point is easy to isolate.
- **Steps:** (1) Build $T$ trees (default 100), each on a random subsample of 256 points: at each node pick a random feature and a random split value in its range; stop when a point is alone (or at a depth limit). (2) For a point $x$, its path length $h_t(x)$ in tree $t$ is the number of cuts until it's isolated. (3) Score $s(x) = 2^{-\bar h(x)/c(n)}$, where $\bar h$ is the average path length over the trees and $c(n)$ the average path length expected for a random point among $n$ (a normalizing constant): close to 1 = anomaly, around 0.5 or below = normal.
- **Why it works:** a point far from the others sits alone in a big empty region, and a random cut falls into that region (and isolates it) with high probability.
- **Caveats:** cuts are axis-aligned, so diagonal structure (like our correlated sensors) is handled roughly; points inside a small dense anomaly cluster are not isolated quickly.
- **Cost:** $O(T \cdot \psi \log \psi)$ to build with subsample size $\psi$, $O(T \log \psi)$ per score: fast, and no distances computed.

In [ ]:
from sklearn.ensemble import IsolationForest
iso = IsolationForest(n_estimators=200, random_state=0).fit(X)       # fit on the normal readings
scores = -iso.score_samples(np.r_[A, X[:3]])     # sklearn returns minus the score: flip so larger = more unusual
print(scores.round(3))                           # P, Q, R, S, T, then three normal readings
# → [0.768 0.766 0.645 0.656 0.662 0.417 0.417 0.418]

P and Q (far from everything) score highest, about 0.77; R, S and T are lower (0.65–0.66) but still clearly above the three normal readings at about 0.42. R and S score lower than under Mahalanobis because the cuts are axis-aligned: they don't see the correlation directly.

## 4. Choosing the threshold

**The problem.** Scores are only half an answer: the line needs a threshold. Too low, and good parts are flagged (false alarms, wasted inspections); too high, and faults slip through.

**The usual recipe without labels.** Decide what false-alarm rate you can live with, say 1%, and set the threshold at that quantile of the scores of **normal** data: `thr = np.quantile(scores_normal, 0.99)`. Then about 1% of good parts are flagged, by construction, whatever the method. That's what §E did. In scikit-learn the same idea is the `contamination` parameter (the expected fraction of anomalies).

**Use held-out normal data if you can.** The training data's own scores are a little optimistic (the model was fitted to them), so on new good parts slightly more than 1% get flagged: §E saw 1.6% instead of 1%.

**With a few labelled anomalies** you can choose the threshold by what matters: the best F1, or the lowest threshold whose precision is still acceptable. §5 shows what to look at.

## 5. Evaluating a detector

**The question.** When a few anomalies are labelled, how good is a detector? Accuracy is useless: with 1% anomalies, "flag nothing" is 99% accurate. Look at the two kinds of mistakes separately.

|  | really an anomaly | really normal |
|---|---|---|
| flagged | TP (caught) | FP (false alarm) |
| not flagged | FN (missed) | TN |

**A worked example.** Ten parts, scored by some detector, 4 of them faulty. Sort by score and lower the threshold one part at a time: each step adds one TP or one FP, which is one point on each of two curves.

🖼 **Picture.** Left: the ten parts sorted by score, with the running counts. Middle: the ROC curve (recall against false positive rate). Right: the precision–recall curve.

In [ ]:
fig_roc_pr_toy()

Read row 3 (threshold just below 0.70, so 4 parts flagged): TP = 3, FP = 1, so recall $= 3/4 = 0.75$, false positive rate $= 1/6 = 0.17$, precision $= 3/4 = 0.75$.

**ROC-AUC has a meaning you can compute by hand. Proposition.** ROC-AUC = the probability that a randomly chosen anomaly scores higher than a randomly chosen normal point (= the fraction of (anomaly, normal) pairs ranked correctly).

**Check on the example.** Anomalies score 0.95, 0.90, 0.70, 0.20; normal parts 0.80, 0.62, 0.55, 0.40, 0.30, 0.10. Pairs ranked correctly: 0.95 beats all 6; 0.90 all 6; 0.70 beats 5 (not 0.80); 0.20 beats 1 (0.10). Total $18$ of $4 \cdot 6 = 24$ pairs: $18/24 = 0.75$, the AUC in the picture. **Why (in words):** each step right on the ROC curve is one normal part passing the threshold; the curve's height there is the fraction of anomalies already above it, i.e. the fraction of anomalies that beat this normal part. The area adds up those heights over all normal parts, each of width $1/N$: the average, over normal parts, of the fraction of anomalies above them.

**Average precision (PR-AUC), by hand:** at each anomaly in the sorted list, the precision so far; then average them. The anomalies are at ranks 1, 2, 4, 9: precisions $1/1, 2/2, 3/4, 4/9$, so AP $= (1 + 1 + 0.75 + 0.444)/4 = 0.799$.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
y_toy = np.array([1, 1, 0, 1, 0, 0, 0, 0, 1, 0]); s_toy = np.array([0.95, 0.90, 0.80, 0.70, 0.62, 0.55, 0.40, 0.30, 0.20, 0.10])
print(roc_auc_score(y_toy, s_toy), round(average_precision_score(y_toy, s_toy), 3))   # → 0.75 0.799

**Which one to report?** ROC divides false positives by the number of **normal** points, which is huge when anomalies are rare, so a detector can have a tiny false-positive rate and still bury every true alarm under false ones. Precision divides by the number of **flagged** points: exactly what the person checking the alarms experiences. **With rare anomalies, report precision–recall (AP) next to ROC-AUC.** Random guessing gives ROC-AUC 0.5 but AP equal to the fraction of anomalies.

## 6. The autoencoder's score, evaluated

Back to §E's factory. Scores of every test part, with the threshold:

🖼 **Picture.** The scores of good test zeros (blue), scratched zeros (orange) and other digits (red), on a log scale; the dashed line is the threshold from §E.

In [ ]:
fig_ae_score_hist()

Wrong parts are far from the good ones (the AE can't make them at all); scratches overlap with good parts, because a thin bar is a small change to a few pixels, and some good zeros are unusually drawn. Now the honest numbers, good vs scratched only:

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
s_good, s_scr = recon_scores(model, good), recon_scores(model, scratched)
y = np.r_[np.zeros(len(s_good)), np.ones(len(s_scr))]             # 880 good (0), 100 scratched (1)
s = np.r_[s_good, s_scr]
flag = s > thr
tp, fp = int(np.sum(flag & (y == 1))), int(np.sum(flag & (y == 0)))
print(tp, fp, round(tp / (tp + fp), 3), round(tp / 100, 3))      # TP, FP, precision, recall
print(round(roc_auc_score(y, s), 3), round(average_precision_score(y, s), 3))
# → 42 14 0.75 0.42
#   0.931 0.655

🖼 **Picture.** The two curves for scratched vs good. ROC-AUC 0.93 sounds excellent; average precision 0.66 says more plainly that catching most scratches costs many false alarms.

In [ ]:
fig_ae_roc_pr()

**Reading it.** At the threshold: 42 scratches caught, 14 false alarms, so 3 out of 4 alarms are real (precision 0.75), and 58 scratches are missed (recall 0.42). Lowering the threshold catches more scratches at the price of more false alarms; the PR curve shows that whole trade-off. **To do better:** a convolutional AE (sharper reconstructions, so scratches stand out more), scoring by the **largest** local error instead of the mean (a scratch is local), or the methods of §7.

## 7. Which method when, and the deep ones

| Method | Score | Good for | Weak when |
|---|---|---|---|
| Gaussian / Mahalanobis (§2) | distance in units of the covariance | a few correlated sensors, one operating mode | several modes or clusters; many features, few samples |
| Gaussian mixture (Week 4) | minus the log-density | several operating modes | high dimension |
| k-nearest-neighbour distance | distance to the k-th nearest normal point | any shape of normal data, low dimension | slow on big data; distances lose meaning in high dimension |
| DBSCAN noise (Week 4) | not in any dense region | exploring a dataset | a yes/no answer, not a score |
| isolation forest (§3) | how few random cuts isolate the point | tabular data, many features, fast | anomalies forming their own dense group |
| autoencoder (§6, `02`) | reconstruction error | images, signals, anything high-dimensional | an AE that generalizes too well rebuilds anomalies too |

**Deep SVDD** (Ruff et al., 2018). Train a network to map every normal point **close to one fixed centre** $c$ in feature space, minimizing $\frac1n \sum_i \lVert \phi(x_i) - c \rVert^2$; the score of a new point is its distance $\lVert \phi(x) - c\rVert$. It's the deep version of a one-class SVM (Week 3's SVM, with only one class): instead of rebuilding the input, it learns features in which normal data is compact. (A detail: the network needs no biases and $c$ must be fixed, otherwise it can map everything to $c$ and score nothing.)

**PatchCore** (Roth et al., 2022), the standard baseline for industrial image inspection (the MVTec AD benchmark). It trains **nothing**: it runs good images through a CNN pretrained on ImageNet (transfer learning, `01`), keeps the middle-layer features of every image patch in a memory bank, and scores a new image's patches by their distance to the nearest stored normal patch. The largest patch distance is the image's score, and the patch distances form a map of where the defect is, like §E's error maps. It's this week's three ideas combined: pretrained features (`01`), normal-only training (`04`), and a localization map (§E).

## 8. Exercises

**Exercise 2 · Mahalanobis scores.** Fit $\mu$ and $S$ on `X_train` `(n, d)` (use `np.cov(X_train.T)`), then return $d_M$ for every row of `X` `(m, d)`: an array `(m,)`. Vectorized: with `D = X - mu`, the row-wise $D_i^\top S^{-1} D_i$ is `np.sum(D @ S_inv * D, axis=1)`.

**Example**: what goes in, and what should come out:
```python
X_train = np.array([[0.0, 0.0], [2.0, 2.0], [2.0, 0.0], [0.0, 2.0]])    # mean (1, 1), S = (4/3) I
mahalanobis_scores(X_train, np.array([[1.0, 1.0], [3.0, 1.0]]))   # → array([0.        , 1.73205081])   2 / sqrt(4/3)
```

In [ ]:
def mahalanobis_scores(X_train, X):
    pass  # your code

In [ ]:
check(2)

**Exercise 3 · The threshold.** Return the `q` quantile of the normal data's scores (`np.quantile`).

**Example**: what goes in, and what should come out:
```python
threshold_at(np.arange(101.0), 0.99)    # → 99.0
```

In [ ]:
def threshold_at(scores_normal, q):
    pass  # your code

In [ ]:
check(3)

**Exercise 4 · Precision and recall.** `y_true` and `y_pred` are 0/1 arrays. Return `(precision, recall)`; when nothing is flagged, precision is 0 (and when there are no anomalies, recall is 0).

**Example**: what goes in, and what should come out:
```python
precision_recall(np.array([1, 1, 0, 0, 1]), np.array([1, 0, 1, 0, 1]))    # → (0.667, 0.667) as floats: TP 2, FP 1, FN 1
```

In [ ]:
def precision_recall(y_true, y_pred):
    pass  # your code

In [ ]:
check(4)

**Exercise 5 · Average precision.** Sort by score, highest first; at each position $k$ compute the precision of the top $k$; return the mean of those precisions at the positions of the true anomalies (as in §5's hand calculation). `np.cumsum` gives the running count of hits.

**Example**: what goes in, and what should come out:
```python
average_precision(np.array([1, 1, 0, 1, 0, 0, 0, 0, 1, 0]),
                  np.array([0.95, 0.90, 0.80, 0.70, 0.62, 0.55, 0.40, 0.30, 0.20, 0.10]))    # → 0.7986   §5's example
```

In [ ]:
def average_precision(y, scores):
    pass  # your code

In [ ]:
check(5)

**Recipe (anomaly detection).** (1) Fit on normal (or mostly normal) data only. (2) Score: Mahalanobis for a few correlated features, isolation forest for tabular data, reconstruction error (or pretrained features) for images and signals. (3) Threshold at the quantile of normal scores that gives the false-alarm rate you can afford. (4) With some labelled anomalies, report precision and recall at the threshold and AP (PR-AUC), not just ROC-AUC. (5) For images, look at the error map: it tells you where.

In [ ]:
for n in range(1, 6):
    check(n)

---
Solutions: `solutions/04_solutions.py`. Next: `05_paper_practice.ipynb`.